# C スタイルフォーマット文字列と str.format は使わず f 文字列で埋め込む

Python には、言語と標準ライブラリに組み込まれた 4種類の異なるフォーマット方式があります。
この項目で述べる1種類を除いては、深刻な欠陥があるのでそれを理解し避ける必要があります。

Python で最もよく使われるフォーマット方式は、フォーマット演算子%を使うものです。

In [37]:
a = 0b10111011
b = 0xc5f
# 2進数 / 16進数値を整数文字列に変換するのに % 演算子を使う
print('Binary is %d, hex is %d' % (a, b))

Binary is 187, hex is 3167


Python を新たに使い始めたプログラマの多くは、慣れていて使いやすいという理由から、この C スタイルフォーマット文字列を使い始めます。
が、Python での C スタイルフォーマット文字列には 4つの問題点があります。

問題点1

フォーマット式の右側のタプルにあるデータ値の型や順序が変わると型変換エラーが起こることです。

In [38]:
# 例えば、以下のようなフォーマットは問題ない
key = 'my_var'
value = 1.234
formatted = '%-10s = %.2f' % (key, value)
print(formatted)

my_var     = 1.23


In [39]:
# しかし、key と value の順序を入れ替えると型変換エラーが起こる
reordered_string = '%-10s = %.2f' % (value, key)  # ここでエラー

TypeError: must be real number, not str

この問題を避けるには、%演算子の両側がきちんとそろっているかを常にチェックする必要があります。
が、変更があるたびにチェックをする必要があります。

問題点2

文字列にフォーマットをする前に値を少し修正しないといけない場合に、読んで理解するのが難しいことです。

In [ ]:
# インラインで変更する前に、台所に貯蔵されている食料品をリストするコード
pantry = [
  ('avocados', 1.25),
  ('bananas', 2.5),
  ('cherries', 15),
]

for i, (item, count) in enumerate(pantry):
    print('#%d: %-10s = %.2f' % (i, item, count))

#0: avocados   = 1.25
#1: bananas    = 2.50
#2: cherries   = 15.00


In [ ]:
# 出力メッセージがもう少し役に立つ値にしようと少し修正を加えると
# フォーマット式が長くなり、より読みにくくなる
for i, (item, count) in enumerate(pantry):
    print('#%d: %-10s = %.2f' % (
        i + 1,
        item.title(),
        round(count)))

#1: Avocados   = 1.00
#2: Bananas    = 2.00
#3: Cherries   = 15.00


問題点3

フォーマット文字列で同じ値を複数回使う場合には、右側のタプルで繰り返し書かないといけないことです。

In [ ]:
template = '%s loves food. See %s cook.'
name = 'Max'
formatted = template % (name, name)
print(formatted)

Max loves food. See Max cook.


In [ ]:
name = 'brad'
# title() をそれぞれつける必要がある
formatted = template % (name.title(), name.title())
print(formatted)

Brad loves food. See Brad cook.


一応、このような問題を回避する手助けとして、Python の %演算子にはタプルではなく辞書でフォーマットする機能があります。
辞書のキーで %(key)のように対応するフォーマット指定子を使います。

In [ ]:
# フォーマット式の右側の値の順序を入れ替えても出力に影響しない
# ので、問題点1は回避可能
key = 'my_var'
value = 1.234

old_way = '%-10s = %.2f' % (key, value)

new_way = '%(key)-10s = %(value).2f' % {'value': value, 'key': key}

reordered = '%(key)-10s = %(value).2f' % {'key': key, 'value': value} # スワップ

assert old_way == new_way == reordered

In [ ]:
# 複数の指定子が同じ値を参照する
# ので、問題点3も回避可能
name = 'Max'

template = '%s loves food. See %s cook.'
before = template % (name, name) # タプル

template = '%(name)s loves food. See %(name)s cook.'
after = template % {'name': name} # 辞書

assert before == after

しかし、辞書を使う方式では、問題点が悪化したり別の問題が生じたりします。
値を少し変更した場合の問題点2では、辞書のキーとコロン演算子が右側に来るので以前よりも見づらくなってしまうことです。

In [ ]:
for i, (item, count) in enumerate(pantry):
  before = '#%d: %-10s = %d' % (
      i + 1,
      item.title(),
      round(count))
  
  after = '#%(loop)d: %(item)-10s = %(count)d' % {
      'loop': i + 1,
      'item': item.title(),
      'count': round(count),
  }

  assert before == after

問題点4

フォーマット式で辞書を使うと、冗長でうるさい感じになります。
キーが、フォーマット指定子と辞書のキーとして1回ずつ、場合によると辞書の値を保持する変数名としてさらに１回指定する必要があります。

In [ ]:
soup = 'lentil'
formatted = 'Today\'s soup is %(soup)s.' % {'soup': soup}
print(formatted)

Today's soup is lentil.


In [ ]:
# 文字が重複するだけでなく、この重複により辞書を使うフォーマット式が長くなる
menu = {
  'soup': 'lentil',
  'oyster': 'kumamoto',
  'special': 'schnitzel',
}

template = ('Today\'s soup is %(soup)s. '
            'buy one get two %(oyster)s oysters!'
            'and our special is %(special)s.')

formatted = template % menu
print(formatted)
            

Today's soup is lentil. buy one get two kumamoto oysters!and our special is schnitzel.


## 組み込みの format と str.format

ここの Python の値を組み込み関数 format に渡すことで使うことができます。

In [ ]:
# 数値を 3桁で区切る「,」
a = 1234.5678
formatted = format(a, ',.2f')
print(formatted)

# センタリングの「^」
b = 'my_string'
formatted = format(b, '^20s')
print('*', formatted, '*')

1,234.57
*      my_string       *


str 型の format メソッドを呼び出して、複数の値をまとめてフォーマットすることもできます。
%d のような Cスタイルフォーマット指定子を使う代わりに {} でプレースホルダの指定ができます。

In [ ]:
key = 'my_var'
value = 1.234

formatted = '{} = {}'.format(key, value)
print(formatted)

my_var = 1.234


In [ ]:
# プレースホルダでは、コロン文字に続けてフォーマット指定することにより、
# 値をどのように文字列にするかを好きなように制御可能
formatted = '{:<10} = {:.2f}'.format(key, value)
print(formatted)

my_var     = 1.23


波括弧の中で format メソッドに渡される引数の位置インデックスを指定し、プレースホルダを置き換える引数を指定することもできます。
こうすると、フォーマット文字列を更新するときに、フォーマット式の右側を同時に変更しないといけないという問題点1が解消される

In [ ]:
formatted = '{1} = {0}'.format(key, value)
print(formatted)

1.234 = my_var


フォーマット文字列では同じ位置インデックスを複数使うことができるので、format メソッドで値を複数個渡す必要がありません。
ので、問題点3が解消される

In [ ]:
formatted = '{0} loves food. See {0} cook.'.format(name)
print(formatted)

Max loves food. See Max cook.


しかし、残念ながら format メソッドでも問題点2,4 の解消はできない

C スタイルフォーマット式の問題点や欠点を考えれば、一般的に str.format メソッドは避けたほうがよいでしょう。
フォーマット指定子に使われるミニ言語（コロンの後の式）や組み込み関数 format の使い方を知っておくことは重要です。しかし、str.format メソッドの残りは歴史的な遺物として、Python の新たな f文字列がどのように動作して、どんな素晴らしいかを理解する助けにしてください。

## フォーマット済み文字列

Python 3.6 は、フォーマット済み文字列（format string）、略して f文字列を導入して上記の問題を解決しました。

In [40]:
key = 'my_var'
value = 1.234

formatted = f'{key} = {value}'
print(formatted)

my_var = 1.234


In [41]:
# f文字列のプレースホルダのコロンの後のミニ言語は、str.format メソッドとほぼ同じです。
formatted = f'{key!r:<10} = {value:.2f}'
print(formatted)

'my_var'   = 1.23


f文字列のフォーマットでは、%演算子を使ったCスタイルフォーマット文字列や str.format メソッドの場合よりも文字数が少なくなります。

In [47]:
# 文字列が少ないことがわかる例
f_string = f'{key:<10} = {value:.2f}'

c_tuple = '%-10s = %.2f' % (key, value)

str_args = '{:<10} = {:.2f}'.format(key, value)

str_kw = '{key:<10} = {value:.2f}'.format(key=key, value=value)

c_dict = '%(key)-10s = %(value).2f' % {'key': key, 'value': value}

assert f_string == c_tuple == str_args == str_kw == c_dict

f文字列ではプレースホルダの波括弧内にすべての Python 式をかけるので、問題点2 が解消され、簡潔な構文でフォーマットされる値にちょっとした修正を施せます。

In [48]:
# Cスタイルフォーマットや st.format メソッドと比べて、f文字列は次の利点があります。
for i, (item, count) in enumerate(pantry):
  old_style = '#%d: %-10s = %d' % (
    i + 1,
    item.title(),
    round(count))
  
  new_style = '#{}: {:<10} = {}'.format(
    i + 1,
    item.title(),
    round(count))
  
  f_string = f'#{i + 1}: {item.title():<10} = {round(count)}'

  assert old_style == new_style == f_string

フォーマット指定子のオプションに Python の式を書くこともできます。

In [49]:
# フォーマット文字列に直接書き込むのではなく、
# 変数を用いて出力する数値の桁数をパラメータ化しています。
places = 3
number = 1.23456
print(f'My number is {number:.{places}f}')

My number is 1.235


## 覚えておくこと

- %演算子を使うCスタイルフォーマット文字列では予想しない動作や読みに管の問題に悩まされる。
- str.format メソッドは、フォーマット指定子のミニ言語に有用な概念を導入しているが、その他ではCスタイルフォーマット文字列の間違いを繰り返しているので避けるべきだ。
- f文字列は、Cスタイルフォーマット文字列の大きな問題を解く新しい構文で、文字列中のフォーマット変数を扱う。
- f文字列は、フォーマット指定子中にどんな Python 式もかけるので、簡潔かつ強力だ。

## Tips 

f文字列の中で出てくる !r は「変換指定子」で、{変数!r}と書くとその変数に repr() を適用するという意味になります。

In [ ]:
# 例
print(f'My number is {key!r}')
# は
print(f'My number is ' + repr(key))
# と同じになります

My number is 'my_var'
My number is 'my_var'


### なぜそんなものがあるのか

値を そのまま見え方のとおり 表示したいときに便利です。
例えば文字列を扱うとき、repr()だと クォート付きで表示される ので違いがわかりやすい。

In [53]:
x = "Hello\nWorld"
print(x)         # 改行される
print(f"{x!r}")  # 'Hello\nWorld' と、そのままの見た目で表示される

Hello
World
'Hello\nWorld'


その他

| 指定子  | 意味              | 実際に使われる関数    |
| ---- | --------------- | ------------ |
| `!r` | repr形式で表示       | `repr(obj)`  |
| `!s` | str形式で表示（デフォルト） | `str(obj)`   |
| `!a` | ASCII形式で表示      | `ascii(obj)` |
